# 05 — Profiling KV cache ON/OFF

    Chạy phiên mới sau notebook 04. Mặc định `RUN_MODE='inspect'`, chỉ xác minh checkpoint. `smoke` đo nhanh checkpoint thử. `official` dùng checkpoint **pretrain chính** trên T4: prompt 128/512/1024/2048, sinh 128 token, 5 lần đo và warmup 2. Output JSON, CSV, PNG, PDF, report.md. Local: `uv sync --group model --group profile --group notebook`. Kaggle cần `matplotlib`; giữ torch CUDA hiện có.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
import torch

REPO_URL = 'https://github.com/ngochhuy/llm-reasoning.git'
REPO_REF = ''  # Kaggle: đặt commit SHA đã review; không dùng main di động.
ON_KAGGLE = Path('/kaggle/working').exists()
REPO_DIR = Path('/kaggle/working/llm-reasoning-repo') if ON_KAGGLE else next(
    (p for p in (Path.cwd(), Path.cwd().parent) if (p/'src'/'llm_reasoning').exists()), Path.cwd())
if ON_KAGGLE:
    if len(REPO_REF) < 40:
        raise ValueError('Đặt REPO_REF bằng commit SHA 40 ký tự đã review trước khi chạy Kaggle')
    if not (REPO_DIR/'src'/'llm_reasoning').exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', REPO_REF], check=True)
CODE_COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
DIRTY = bool(subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip())
print('commit:', CODE_COMMIT, 'dirty:', DIRTY, 'torch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
sys.path.insert(0, str(REPO_DIR/'src'))
os.environ['PYTHONPATH'] = str(REPO_DIR/'src') + os.pathsep + os.environ.get('PYTHONPATH', '')

In [ ]:
from llm_reasoning.checkpoint import file_sha256, load_model_artifact
from llm_reasoning.tokenizer import ByteBPETokenizer
RUN_MODE = 'inspect'  # inspect | smoke | official
APPROVED_EARLY_STOP = False  # chỉ bật sau review checkpoint dừng sớm vì giới hạn thời gian.
CHECKPOINT = Path('/kaggle/input/llm-reasoning-pretrain/latest.pt') if ON_KAGGLE else REPO_DIR/'artifacts/nb04_pretrain/latest.pt'
TOKENIZER = Path('/kaggle/input/llm-reasoning-data/pretrain/tokenizer.json') if ON_KAGGLE else REPO_DIR/'artifacts/data/pretrain/tokenizer.json'
OUTPUT = Path('/kaggle/working/kv_profile') if ON_KAGGLE else REPO_DIR/'artifacts/nb05_profile'
if RUN_MODE not in ('inspect', 'smoke', 'official'):
    raise ValueError('RUN_MODE không hợp lệ')
if RUN_MODE == 'official' and (not ON_KAGGLE or not torch.cuda.is_available() or DIRTY):
    raise RuntimeError('Profiling chính thức yêu cầu T4/CUDA và commit sạch')
_, meta = load_model_artifact(CHECKPOINT, tokenizer_path=TOKENIZER)
if meta['stage'] != 'pretrain':
    raise ValueError('Notebook 05 yêu cầu checkpoint pretrain')
if RUN_MODE == 'official' and meta['checkpoint_kind'] != 'training':
    raise ValueError('Checkpoint này chỉ là smoke/unclassified, không dùng đo chính thức')
if RUN_MODE == 'official' and not APPROVED_EARLY_STOP and meta['train_state']['attempted_steps'] < meta['total_steps']:
    raise ValueError('Checkpoint pretrain còn dở; hoàn thành hoặc ghi rõ đây là đo pilot')
print('Checkpoint:', CHECKPOINT, 'SHA:', file_sha256(CHECKPOINT))
print('Tokenizer SHA:', file_sha256(TOKENIZER), 'metadata:', meta)

In [ ]:
if RUN_MODE != 'inspect':
    if OUTPUT.exists():
        raise FileExistsError('Output profiling tồn tại; chọn thư mục mới')
    command = [sys.executable, '-m', 'llm_reasoning.profiling', '--checkpoint', str(CHECKPOINT),
               '--tokenizer', str(TOKENIZER), '--output', str(OUTPUT),
               '--device', 'cuda' if RUN_MODE == 'official' else 'cpu',
               '--dtype', 'float16' if RUN_MODE == 'official' else 'checkpoint',
               '--prompt-lengths', *map(str, (128, 512, 1024, 2048) if RUN_MODE == 'official' else (16, 32)),
               '--new-tokens', '128' if RUN_MODE == 'official' else '8',
               '--repetitions', '5' if RUN_MODE == 'official' else '2',
               '--warmup', '2' if RUN_MODE == 'official' else '1']
    subprocess.run(command, cwd=REPO_DIR, check=True)
    result = json.loads((OUTPUT/'results.json').read_text(encoding='utf-8'))
    print('Greedy pairs match:', result['all_pairs_match'], 'report:', OUTPUT/'report.md')
    if not result['all_pairs_match']:
        raise RuntimeError('KV cache ON/OFF không khớp')
else:
    print('Inspect hoàn tất; chưa chạy đo.')